# IAN 601 — Module 14: Machine Learning with House Prices

Machine learning uses examples to estimate a relationship that can be applied to new cases.

This notebook is a standalone introduction to supervised machine learning through a house-price regression problem. We begin with a familiar algebraic relationship, add realistic noise, then move to a CSV dataset with multiple numeric and categorical features.

**Main path:** algebra → data → features → train/test split → regression → prediction → evaluation → pipeline

## Learning goals

By the end of this notebook, you should be able to:

1. connect a linear equation with a fitted linear model;
2. explain why real data contains noise;
3. load and inspect a modeling dataset;
4. distinguish features from a target;
5. represent one observation as a feature vector;
6. explain why categorical features require numerical encoding;
7. separate data into training and test sets;
8. fit a linear regression model;
9. compare actual and predicted values;
10. interpret MAE, RMSE, and R² at an introductory level;
11. predict a new house price;
12. explain data leakage;
13. explain why preprocessing and modeling can be combined in a pipeline;
14. distinguish regression from classification.

## How to use this notebook

Run the cells from top to bottom the first time so that variables and imported libraries are available when later cells need them.

As you work:

1. read the explanation before the code;
2. predict the result;
3. run the cell;
4. inspect the output carefully;
5. change one value, condition, or option;
6. rerun the cell and explain what changed.

For notebooks that load a CSV from GitHub, the dataset is loaded automatically. If you restart the runtime, run the notebook again from the beginning.

## 1. Start with algebra: y = 2x + 1

When the relationship is known, prediction is simple.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

x = np.arange(0, 11)
y = 2 * x + 1

pd.DataFrame({"x": x, "y": y})

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(x, y, marker="o")
plt.title("Known Relationship: y = 2x + 1")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

## 2. Reverse engineer the relationship

If we only have the x and y values, a linear model can estimate the slope and intercept.

In [ ]:
from sklearn.linear_model import LinearRegression

X_simple = x.reshape(-1, 1)

simple_model = LinearRegression()
simple_model.fit(X_simple, y)

print("Estimated slope:", simple_model.coef_[0])
print("Estimated intercept:", simple_model.intercept_)

## 3. Real data is noisy

Real-world measurements rarely fall perfectly on one line.

In [ ]:
rng = np.random.default_rng(601)

x_noisy = np.arange(0, 30)
y_noisy = 2 * x_noisy + 1 + rng.normal(0, 5, size=len(x_noisy))

plt.figure(figsize=(7, 4))
plt.scatter(x_noisy, y_noisy)
plt.title("Noisy Observations")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

## 4. Best-fit line

The model estimates the line that best summarizes the relationship in the noisy data.

In [ ]:
model_noisy = LinearRegression()
model_noisy.fit(x_noisy.reshape(-1, 1), y_noisy)

predicted_noisy = model_noisy.predict(x_noisy.reshape(-1, 1))

plt.figure(figsize=(7, 4))
plt.scatter(x_noisy, y_noisy, label="Observed")
plt.plot(x_noisy, predicted_noisy, label="Best-fit line")
plt.title("Observed Data and Best-Fit Line")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

print("Estimated slope:", round(model_noisy.coef_[0], 3))
print("Estimated intercept:", round(model_noisy.intercept_, 3))

## 5. Load the house-price CSV

Now we move from a simple mathematical example to a realistic tabular dataset.

In [ ]:
data_url = "https://raw.githubusercontent.com/ali2449/IAN-601-Introduction-to-Data-Analytics-Methods-and-Approaches/main/IAN601_Module14_House_Prices_600.csv"

houses = pd.read_csv(data_url)

houses.head()

## 6. Inspect the housing data

The target is **sale_price**. Everything else that is available before the sale can potentially be used as a feature.

In [ ]:
print("Shape:", houses.shape)
print()
print(houses.dtypes)
print()
print("Missing values:")
print(houses.isna().sum())

In [ ]:
houses.describe(include="all").T

## 7. Start with one feature: square feet

This is the easiest connection to the algebra idea.

We ask:

> Can square footage alone help predict sale price?

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(
    houses["square_feet"],
    houses["sale_price"],
    alpha=0.55
)
plt.title("Square Feet vs Sale Price")
plt.xlabel("Square feet")
plt.ylabel("Sale price")
plt.show()

## 8. Fit a one-feature linear regression model

In [ ]:
X_one = houses[["square_feet"]]
y_price = houses["sale_price"]

one_feature_model = LinearRegression()
one_feature_model.fit(X_one, y_price)

print("Slope:", round(one_feature_model.coef_[0], 2))
print("Intercept:", round(one_feature_model.intercept_, 2))

## 9. Draw the regression line

In [ ]:
x_line = np.linspace(
    houses["square_feet"].min(),
    houses["square_feet"].max(),
    100
).reshape(-1, 1)

y_line = one_feature_model.predict(x_line)

plt.figure(figsize=(7, 4))
plt.scatter(
    houses["square_feet"],
    houses["sale_price"],
    alpha=0.35,
    label="Observed houses"
)
plt.plot(
    x_line,
    y_line,
    linewidth=2,
    label="Regression line"
)

plt.title("One-Feature House Price Model")
plt.xlabel("Square feet")
plt.ylabel("Sale price")
plt.legend()
plt.show()

## 10. Predict the price of a new house using one feature

In [ ]:
new_square_feet = pd.DataFrame({
    "square_feet": [1800]
})

predicted_price = one_feature_model.predict(new_square_feet)[0]

print(f"Predicted price for 1,800 sq. ft.: ${predicted_price:,.0f}")

## 11. One feature is useful, but incomplete

House price depends on more than square footage.

We can add:

- bedrooms
- bathrooms
- age
- lot size
- distance to the city center
- neighborhood
- garage
- condition

## 12. Features become a numeric vector

For one house, numeric features can be thought of as a vector.

In [ ]:
numeric_features = [
    "square_feet",
    "bedrooms",
    "bathrooms",
    "age_years",
    "lot_size_acres",
    "distance_to_center_miles"
]

houses[numeric_features].head()

In [ ]:
first_house_vector = houses.loc[0, numeric_features].to_numpy()

print(first_house_vector)

## 13. What about text features?

Machine learning models need numbers.

Categorical columns such as neighborhood, garage, and condition can be converted with one-hot encoding.

In [ ]:
categorical_preview = pd.get_dummies(
    houses[["neighborhood", "garage", "condition"]],
    dtype=int
)

categorical_preview.head()

## 14. Define X and y

- **X** contains the input features.
- **y** contains the target we want to predict.

In [ ]:
feature_columns = [
    "square_feet",
    "bedrooms",
    "bathrooms",
    "age_years",
    "lot_size_acres",
    "distance_to_center_miles",
    "neighborhood",
    "garage",
    "condition"
]

X = houses[feature_columns]
y = houses["sale_price"]

print("X shape:", X.shape)
print("y shape:", y.shape)

## 15. Train/test split

We should not evaluate the model on the same rows used to train it.

The test set represents houses the model did not see during training.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=601
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## 16. Build preprocessing and regression together

A pipeline keeps the preprocessing steps and model together.

Numeric columns are standardized. Categorical columns are one-hot encoded.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

numeric_cols = [
    "square_feet",
    "bedrooms",
    "bathrooms",
    "age_years",
    "lot_size_acres",
    "distance_to_center_miles"
]

categorical_cols = [
    "neighborhood",
    "garage",
    "condition"
]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

house_model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", LinearRegression())
    ]
)

house_model

## 17. Train the house-price model

In [ ]:
house_model.fit(X_train, y_train)

print("Model trained")

## 18. Predict prices for the test set

In [ ]:
predicted_prices = house_model.predict(X_test)

comparison = pd.DataFrame({
    "actual_price": y_test.to_numpy(),
    "predicted_price": predicted_prices
})

comparison.head(10).round(0)

## 19. Actual vs predicted prices

A good regression model should place many points close to the diagonal line.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(
    y_test,
    predicted_prices,
    alpha=0.65
)

minimum = min(y_test.min(), predicted_prices.min())
maximum = max(y_test.max(), predicted_prices.max())

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--"
)

plt.title("Actual vs Predicted House Prices")
plt.xlabel("Actual price")
plt.ylabel("Predicted price")
plt.show()

## 20. Evaluate regression

Three common regression metrics are:

- **MAE** — average absolute prediction error
- **RMSE** — gives more weight to large errors
- **R²** — how much of the variation is explained by the model

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, predicted_prices)
rmse = np.sqrt(
    mean_squared_error(y_test, predicted_prices)
)
r2 = r2_score(y_test, predicted_prices)

print(f"MAE:  ${mae:,.0f}")
print(f"RMSE: ${rmse:,.0f}")
print(f"R²:   {r2:.3f}")

## 21. Predict a completely new house

In [ ]:
new_house = pd.DataFrame({
    "square_feet": [1800],
    "bedrooms": [3],
    "bathrooms": [2],
    "age_years": [10],
    "lot_size_acres": [0.30],
    "distance_to_center_miles": [7.5],
    "neighborhood": ["West"],
    "garage": ["Yes"],
    "condition": ["Good"]
})

new_prediction = house_model.predict(new_house)[0]

print(f"Predicted sale price: ${new_prediction:,.0f}")

## 22. Compare a simple model with the full model

This helps students see why adding useful features can improve prediction.

In [ ]:
from sklearn.metrics import mean_absolute_error

# One-feature model using only square_feet
simple_train = X_train[["square_feet"]]
simple_test = X_test[["square_feet"]]

simple_house_model = LinearRegression()
simple_house_model.fit(simple_train, y_train)

simple_predictions = simple_house_model.predict(simple_test)

simple_mae = mean_absolute_error(
    y_test,
    simple_predictions
)

full_mae = mean_absolute_error(
    y_test,
    predicted_prices
)

print(f"Square-feet-only MAE: ${simple_mae:,.0f}")
print(f"Full model MAE:       ${full_mae:,.0f}")

## 23. Data leakage

A feature should only contain information that would actually be available when the prediction is made.

For example, do **not** use a value that is calculated after the house is sold to predict sale price.

Leakage can make a model look much better than it really is.

## 24. Why use a pipeline?

The pipeline helps ensure that:

1. the same preprocessing is applied during training and prediction,
2. categorical features are handled consistently,
3. preprocessing stays connected to the model,
4. future code is easier to reuse.

## 25. Regression versus classification

This module focuses mainly on regression.

- **Regression:** predict a number  
  Example: house price

- **Classification:** predict a category  
  Example: whether a customer will buy

The next machine learning course can explore classification metrics and algorithms in much more depth.

## 26. Machine learning workflow

A useful workflow to remember:

1. Define the question.
2. Load and inspect the data.
3. Choose the target.
4. Choose features.
5. Separate X and y.
6. Split training and testing data.
7. Preprocess features.
8. Train a baseline or simple model.
9. Train a better model.
10. Evaluate on unseen test data.
11. Predict new cases.
12. Check for leakage and limitations.

## Practice challenge

1. Train a model using only `square_feet` and `bedrooms`.
2. Compare its MAE with the full model.
3. Predict the price of a 2,200 sq. ft. house.
4. Change the neighborhood and compare the prediction.
5. Remove `condition` and retrain the model.
6. Explain in your own words what the train/test split protects us from.
7. Give one example of data leakage in a house-price project.

In [ ]:
# Practice area




## Summary and self-check

The central machine-learning workflow is:

**features → training → prediction → evaluation**

Before finishing, make sure you can explain:

- what `X` and `y` represent;
- why the test set must remain separate from model training;
- why categorical features need encoding;
- what a prediction error means;
- how MAE differs conceptually from RMSE;
- what R² summarizes;
- why a pipeline is useful;
- what data leakage is;
- how regression differs from classification.

The most important idea is not a specific algorithm. It is learning how to build and evaluate a model without using information the model should not have.